<a href="https://colab.research.google.com/github/spbromberg/ds2002-fa26/blob/main/2026-09-11%20%E2%80%94%20SQL%20Challenge%20Set%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q1 = q('''
SELECT tracks.track_id,
       tracks.title,
       artists.name AS artist_name,
       artists.country
FROM tracks
JOIN artists ON tracks.artist_id = artists.artist_id
ORDER BY tracks.track_id;
''')
q1

# The JOIN connects each track to its artist so to show the artist's name and country.

,track_id,title,artist_name,country
0,10,Skyline,Nova Waves,US
1,11,Undertow,Nova Waves,US
2,12,Foothills,The Blue Ridge,US
3,13,Aurora,Kestrel,UK
4,14,Nightfall,Kestrel,UK
5,15,Sol,Marisol,ES
6,16,Coastline,The Blue Ridge,US
7,17,Ridgeline,The Blue Ridge,US
8,18,Untitled Demo,Kestrel,UK


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q2 = q('''
SELECT genre,
       AVG(seconds) AS average_seconds
FROM tracks
WHERE genre IS NOT NULL
GROUP BY genre
ORDER BY average_seconds DESC;
''')
q2

# GROUP BY makes one row per genre, and AVG calculates each genre's average length.
# Sorting from largest to smallest puts the genre with the longest average first.

,genre,average_seconds
0,Electronic,287.5
1,Pop,220.5
2,Latin,210.0
3,Folk,203.0


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q3 = q('''
SELECT user,
       COUNT(*) AS plays,
       COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
ORDER BY user;
''')
q3

# COUNT(*) counts all plays, while COUNT(DISTINCT track_id) counts different tracks.
# GROUP BY user gives one summary row for each user.

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q4 = q('''
SELECT tracks.track_id,
       tracks.title
FROM tracks
LEFT JOIN plays ON tracks.track_id = plays.track_id
WHERE plays.track_id IS NULL
ORDER BY tracks.track_id;
''')
q4

# LEFT JOIN keeps every track, even when it has no matching play.
# The NULL check keeps only tracks that were never played.

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q5 = q('''
SELECT artists.name AS artist_name,
       SUM(tracks.seconds) AS total_seconds,
       ROUND(SUM(tracks.seconds) / 60.0, 1) AS minutes
FROM plays
JOIN tracks ON plays.track_id = tracks.track_id
JOIN artists ON tracks.artist_id = artists.artist_id
GROUP BY artists.name
ORDER BY total_seconds DESC;
''')
q5

# The joins connect each play to its track and then to its artist.
# SUM adds the length of every played track, and the result is sorted from largest to smallest.

,artist_name,total_seconds,minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [ ]:
q6 = q('''
SELECT track_id,
       title
FROM tracks
WHERE genre IS NULL
ORDER BY track_id;
''')
q6

# IS NULL finds tracks with no genre value.
# WHERE genre != 'Pop' would not return these rows because NULL is not equal to 'Pop' or to any other value in SQL.
# In SQL, comparisons against NULL are unknown, so rows with NULL are filtered out instead of being treated as a match.


,track_id,title
0,18,Untitled Demo


### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q7 = q('''
SELECT played_on,
       COUNT(*) AS plays,
       COUNT(DISTINCT user) AS distinct_users
FROM plays
GROUP BY played_on
ORDER BY played_on;
''')
q7

# GROUP BY played_on creates one row per date.
# COUNT(*) counts plays, and COUNT(DISTINCT user) counts different active users.

,played_on,plays,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

Q4 was the most tricky, because I first thought an inner JOIN would work. But, an inner JOIN only keeps tracks that have a matching row in the plays table, so it would remove tracks that were never played. I used a LEFT JOIN instead, which keeps every track, and then filtered for rows where there were null values. This returned the two tracks that have never been played.